In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yaml
import os
import glob

## Input Parameters

In [ ]:
# Path to the rate_coefficient.log file
log_path = '/Users/shikim/code/pynta_local/pynta/IPython/rate_coefficient.log'

# (Optional) path to reaction yaml for human-readable labels; set to None to skip
yaml_path = None

# Select which TS directories to plot, e.g. ['TS0', 'TS2']; set to [] or None to include all
selected_ts = ['TS3', 'TS2']

# Temperature range (K)
T = np.linspace(500, 1400, 50)

## Load Data

In [ ]:
def parse_surface_arrhenius(text):
    a_str    = text.split('A=(')[1].split(')')[0]
    n_str    = text.split(', n=')[1].split(',')[0].strip()
    ea_str   = text.split('Ea=(')[1].split(')')[0]
    A_val    = float(a_str.split(',')[0])
    A_units  = a_str.split(',', 1)[1].strip().replace(chr(39), '')
    n_val    = float(n_str)
    Ea_val   = float(ea_str.split(',')[0])
    Ea_units = ea_str.split(',', 1)[1].strip().replace(chr(39), '')
    Ea_kJ    = Ea_val if 'kJ' in Ea_units else Ea_val / 1000.0
    return A_val, n_val, Ea_kJ, A_units


def parse_rate_coefficient_log(path):
    """Parse rate_coefficient.log into dict: ts_name -> list of entry dicts."""
    results = {}
    current_ts = None
    current_entry = {}

    def _save(ts, entry):
        if ts is not None and entry:
            results.setdefault(ts, []).append(entry)

    with open(path) as f:
        for line in f:
            stripped = line.strip()
            if stripped.startswith('===') and stripped.endswith('==='):
                _save(current_ts, current_entry)
                current_ts = stripped.strip('=').strip()
                current_entry = {}
            elif stripped.startswith('Index:'):
                _save(current_ts, current_entry)
                current_entry = {'index': stripped.split('Index:', 1)[1].strip()}
            elif stripped.startswith('Reaction:'):
                current_entry['reaction'] = stripped.split('Reaction:', 1)[1].strip()
            elif stripped.startswith('arr_f:'):
                val = stripped.split('arr_f:', 1)[1].strip()
                current_entry['arr_f'] = None if val == 'None' else parse_surface_arrhenius(val)
            elif stripped.startswith('arr_r:'):
                val = stripped.split('arr_r:', 1)[1].strip()
                current_entry['arr_r'] = None if val == 'None' else parse_surface_arrhenius(val)
    _save(current_ts, current_entry)
    return results


# Load yaml labels if provided
yaml_reactions = {}
if yaml_path:
    with open(yaml_path) as f:
        yaml_reactions = {e['reaction']: e for e in yaml.safe_load(f)}

# Parse log and select TS entries with lowest forward Ea
all_ts_entries = parse_rate_coefficient_log(log_path)
ts_names = selected_ts if selected_ts else sorted(all_ts_entries.keys())

ts_data = {}
for ts_name in ts_names:
    entries = all_ts_entries.get(ts_name, [])
    if not entries:
        print('{}: not found in log'.format(ts_name))
        continue
    valid = [e for e in entries if e.get('arr_f') is not None]
    if not valid:
        print('{}: no valid arr_f entries'.format(ts_name))
        continue
    best = min(valid, key=lambda e: e['arr_f'][2])
    ts_data[ts_name] = best
    print('{}: {}  (Ea_f = {:.1f} kJ/mol)'.format(ts_name, best.get('reaction', 'unknown'), best['arr_f'][2]))

## Calculate Rate Coefficients

In [ ]:
R = 0.0083145  # kJ/(mol*K)

records_f, records_r = [], []

for ts_name, data in ts_data.items():
    rxn_str = data.get('reaction', ts_name)
    label   = yaml_reactions.get(rxn_str, {}).get('reaction', rxn_str)

    if data.get('arr_f') is not None:
        A, n, Ea_kJ, A_units = data['arr_f']
        k = A * (T ** n) * np.exp(-Ea_kJ / (R * T))
        records_f.append({'label': label, 'k': k, 'A_units': A_units})

    if data.get('arr_r') is not None:
        A, n, Ea_kJ, A_units = data['arr_r']
        k = A * (T ** n) * np.exp(-Ea_kJ / (R * T))
        records_r.append({'label': label, 'k': k, 'A_units': A_units})

## Plot Rate Coefficients

In [ ]:
tableau_colors = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
    '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'
]
line_thickness = 4


def style_ax(ax, ylabel):
    ax.set_yscale('log')
    ax.set_xlabel('Temperature (K)', fontsize=15, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=15, fontweight='bold')
    ax.set_xlim(T[0], T[-1])
    ax.tick_params(axis='both', which='major', labelsize=13, width=2)
    for tick in ax.xaxis.get_major_ticks():
        tick.label1.set_fontweight('bold')
    for tick in ax.yaxis.get_major_ticks():
        tick.label1.set_fontweight('bold')
    for spine in ax.spines.values():
        spine.set_linewidth(3)
    ax.grid(False)
    ax.legend(fontsize=10, loc='best')


# Forward rate coefficients
fig, ax = plt.subplots(figsize=(8, 7))
for i, rec in enumerate(records_f):
    ax.plot(T, rec['k'], label=rec['label'],
            color=tableau_colors[i % len(tableau_colors)], linewidth=line_thickness)
style_ax(ax, 'Rate Coefficient k (forward)')
plt.tight_layout()
plt.show()


# Reverse rate coefficients
if records_r:
    fig, ax = plt.subplots(figsize=(8, 7))
    for i, rec in enumerate(records_r):
        ax.plot(T, rec['k'], label=rec['label'],
                color=tableau_colors[i % len(tableau_colors)],
                linewidth=line_thickness, linestyle='--')
    style_ax(ax, 'Rate Coefficient k (reverse)')
    plt.tight_layout()
    plt.show()